# 연구 자료와 추가 통제 실험의 관계
**해결책 — E11 → E14 → E16의 연구 흐름을 기준으로 읽기**

이 파일은 공유된 Colab의 **실행 완료 코드와 저장 출력을 유지**하고, 논문 해석 메모만 추가한 사본입니다. 편집 과정에서 모델을 다시 학습하지 않았습니다. 코드 셀은 원본과 동일합니다.

## 이 노트북이 대신하지 않는 연구

E11은 전체 은닉 표현에서 K/C 4-state를 판독했고, E13은 probe와 관련된 구성요소를 분석했습니다. E14는 헤드 수준 판별과 5-fold 재현성을, E16은 소량 라벨로 역할별 헤드를 찾고 판독하는 가능성을 조사했습니다.

사용자가 제공한 E16 기록에서 `diagnostic_same_roleheads_LR`의 Macro-F1은 **0.6120±0.0123**이고, probe 없는 통계량 규칙의 F1은 **0.3818**입니다. 이 두 값은 서로 다른 판독기의 결과입니다. 소량 라벨로 헤드를 탐색할 수 있다는 관찰과 robust probe-free routing의 성공을 같은 주장으로 쓰지 않습니다.

## 이번 추가 실험의 위치

이 노트북은 **E14의 4-state 순위·가변 k 헤드 선택 경로**를 고정하고 라벨 예산만 줄인 통제 실험입니다. E16의 역할별 탐색 파이프라인을 예산별로 재실행한 실험이 아닙니다.

| 구분 | 질문 | 결과 및 해석 |
|---|---|---|
| E16 기존 관찰 | 소량 라벨로 역할별 헤드를 찾고 상태를 판독할 수 있는가? | LR 0.6120±0.0123, 규칙 0.3818 — 제공 연구기록 기준 |
| 추가 E14 경로 통제 | 같은 평가 질문·헤드 선택 경로에서 예산만 줄이면 얼마나 달라지는가? | 80% 0.6308, 20% 0.5914±0.0170; Δ=-0.0395, 조건부 95% CI [-0.0632,-0.0159] |
| 후속 독립 평가 | 헤드·판독기/규칙을 완전히 고정해도 새 데이터에서 동작하는가? | 아래 후속 설계에 해당하며 아직 실행된 결과가 아님 |

따라서 **0.5914를 E16의 0.6120에 대한 수정값으로 쓰지 않습니다.** 또한 이번 결과만으로 E16 역할별 탐색의 20% 대 80% 차이를 추정할 수 없습니다. E16 자체의 라벨 효율성에 대한 직접 주장은 그 경로를 고정한 별도 비교가 필요합니다.

## 논문에 연결할 핵심

1. **판별 정보:** 전체 은닉 표현과 헤드 벡터에서 상태를 읽을 수 있는가?
2. **소량 라벨:** 적은 라벨로 판독이 가능한가? 예산 절감 시 실제 성능 손실은 얼마인가?
3. **판독과 개입의 분리:** 좋은 4-state probe가 곧 좋은 retain/accept/no-op 라우터는 아니다.
4. **고정 후 독립 평가:** 헤드 ID·개수, 정규화, 판독기 또는 통계량 규칙, 임계값, 개입 방향·강도를 개발 자료에서 고정한 뒤 새 질문·엔티티에서 평가한다. 같은 p8s를 새로 분할하는 것만으로 독립성이 확보되지는 않는다.

`probe를 재학습하지 않음`은 `실행 시 probe를 호출하지 않음`과 다릅니다. 고정 probe를 사용하는 조건과 probe-free 규칙 조건을 분리하고, QA에서는 같은 개입 후보 아래 무개입·일괄 개입·상태 기반 개입을 비교합니다. 상태 라벨은 test-time selector에 넣지 않습니다.

---
아래 셀의 제작 당시 사전 점검 안내와, 이후 저장된 실제 실행 결과를 구분하세요. **실제 실험값은 실행 출력 및 생성 CSV**에 있습니다.


# KSC-LLB — 같은 E14 파이프라인의 라벨 예산 통제·논문 검증 실험
**해결책 | 산출물 ① 실험 설계서 + 실행 노트북**

박세령 · 손현경 · 이다영 / 이화여자대학교 · 지도교수 황의원  
연락 이메일: 03.ryeong@ewhain.net

이 노트북은 **4-state 상태 판별**의 주장을 검증한다. `80% vs 20%`에서 모델·특징·평가 질문·헤드 선택·Probe 학습을 그대로 두고, 공개하는 개발 라벨만 바꾼다. 실제 E14 원본의 `StandardScaler → LogisticRegression(C=0.1)` 및 `fit → utility_fit → policy_cal → test`를 확인하여 작성했다. E16의 다른 role-head 방식으로 교체하지 않았다.

**사용법:** Colab 또는 Jupyter에서 위에서 아래로 실행한다. 기존 공유된 E14 자료 4개를 자동 조회·다운로드하고 SHA-256을 검사한다. LLM·GPU·Hugging Face 토큰·추가 데이터 업로드는 필요하지 않다. Colab Drive 인증 시 기존 p8s 자료를 가진 계정으로 연결한다. 공유 권한이 없으면 임의의 다른 파일로 대체하지 않고 중단한다.

**실행 확인 범위:** 원 데이터 1,224행의 분할·라벨 감사, fold 0의 20/80 학습 및 모든 코드 대조군의 축소 실행, 부트스트랩 불변성 검사를 수행했다. 제작 환경 scikit-learn 1.8.0에서 수행한 코드 검증이며, 아래 정식 환경 1.6.1의 5-fold×5시드 결과가 아니다. **새 논문 결과는 전체 실행 후 생성되는 CSV에서 가져온다.** 이전 E14/E16 점수를 새 비교의 결과로 재사용하지 않는다.

## 0. 문제상황 → 원인 → 해결책 → 얻을 근거

**문제상황:** E14 5-fold head-set 0.6306과 E16 low-label 0.6120±0.0123은 분할과 head 선택 방식이 다르다. 숫자 차이를 라벨 양의 영향으로만 해석할 수 없다.

**원인:** 평가 질문, 헤드 집합을 정하는 방법, 사용 가능한 라벨 예산이 함께 달랐다. E14의 지도학습 head-set F1과 runtime probe-free 규칙의 F1 역시 다른 결과다.

**해결책:** E14의 감독학습 판독 경로만 고정한다. 원본의 `HEAD_LAST`(L13, o_proj 직전, 마지막 프롬프트 토큰)와 `BLOCK_LAST`(L13 블록 출력)를 재사용한다. 모든 low-budget 조건을 동일한 80% 조건과 대응시켜 평가한다.

**얻을 근거:** 라벨 예산 감소에 따른 F1 변화량, 헤드 선발의 이득, magnitude만으로는 설명되지 않는 판별 신호, 표면 단서·라벨 누출 대조를 분리하여 보고한다. 좋은 결과가 나오는 것을 보장하지 않으며, 실패한 주장도 결과에 남긴다.

### 논문 주장과 실험의 연결
| ID | 확인할 주장 | 비교·통제 | 논문에 쓸 수 있는 범위 |
|---|---|---|---|
| P0 | 원 데이터와 E14 경로를 정확히 사용했다 | 파일 해시, 라벨/row/fold 일치, 80% 재실행과 저장 OOF 대조 | 재실행 조건과 수치 차이를 공개 |
| P1 (주 실험) | 라벨 예산을 줄이면 판별력은 얼마나 변하는가 | 20% vs 80%, 같은 5-fold 및 5개 라벨 추출 시드 | ΔF1와 질문 단위 조건부 CI; 동등성 자동 주장 금지 |
| P2 | 헤드 선택이 임의 선택보다 유용한가 | 상위 4개 vs 무작위 4개 20회; 전32개 head 벡터도 제시 | 고정 k=4의 선발 이득; 가변 head-set 전체로 일반화 금지 |
| P3 | 벡터 크기 외 정보가 도움이 되는가 | 같은 상위4개 raw vector / L2-unit vector / norm + 동일 LR | 고정된 헤드의 판독 방식 비교; 인과적 방향 메커니즘 단정 금지 |
| P4 | 단순 표면 특징 또는 라벨 의존적 누출인가 | question-only / question+context TF-IDF; 개발 라벨 쌍단위 셔플 후 전체 선택 재실행 | 텍스트 단서·negative control 결과를 보고; 정식 순열 p값 아님 |
| P5 (새 자료 필요) | 처음 보는 분포에도 적용되는가 | 본 자료와 독립된 질문/엔티티의 새 cohort, decoder·헤드 고정 | 실제 외부 결과가 있을 때만 일반화 주장 |

P0–P4는 아래에 실행 코드가 있다. P5 및 실제 QA/steering 개선은 아래 마지막 절에 **별도 검증 설계**로만 둔다. 실행하지 않은 P5 또는 QA 성능을 표에 채우지 않는다.

## 1. 정식 실행 설정 — 결과를 보기 전에 고정

주 비교는 **전체 612개 질문 대비** 20%인 122개의 개발 라벨과, 각 fold의 나머지 전부인 489/490개의 개발 라벨이다. 80% 조건도 다시 학습한다. 동일 평가 질문 122/123개에 대해 점수를 계산한다.

fold 0 예시: 80%는 `fit 366 / utility_fit 61 / policy_cal 62`, 20%는 `fit 92 / utility_fit 15 / policy_cal 15`개 질문이다. 두 조건의 test는 같은 123개 질문이다. 라벨 예산에는 **헤드 순위·개수 선택에서 본 라벨**도 포함된다. 남은 low-budget 개발 질문은 쓰지 않는다.

기존 benchmark split은 K로 층화되어 있다. 이 고정 분할을 바꾸지 않되, **추가 low-budget 샘플링에는 K/C 라벨을 보지 않는다.** 원래 역할별 질문을 시드로 섞고 nested prefix만 사용한다. 시드나 표본을 성능에 맞게 고르지 않는다. 클래스가 빠진 조건은 보고하고 중단하며 재추첨하지 않는다.

**20%는 한 번의 학습·개발 과정의 라벨 접근 예산**이다. 여러 fold/시드의 라벨 합집합이나 평가 라벨까지 포함한 총 주석 비용이 20%라는 뜻이 아니다. 비교 시드를 늘려도 독립 평가 질문은 612개다.

`noninferiority_margin=None`이 기본값이다. 허용 가능한 성능 손실을 운영상 근거로 먼저 정한 경우에만 바꾼다. 이전의 0.0181 차이를 보고 편리한 허용치를 정하면 안 된다. 신뢰구간이 0을 포함하는 것만으로 동등하다고 쓰지 않는다.

In [ ]:
# 환경 준비: 최초 실행 셀. sklearn을 먼저 import하지 마세요.
import sys, subprocess, importlib.metadata
try:
    installed = importlib.metadata.version('scikit-learn')
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed != '1.6.1':
    if 'sklearn' in sys.modules:
        raise RuntimeError('scikit-learn이 이미 import되어 있습니다. 런타임 재시작 후 이 셀부터 실행하세요.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn==1.6.1'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gdown', 'threadpoolctl', 'pandas', 'matplotlib'])
print('scikit-learn', importlib.metadata.version('scikit-learn'))

In [ ]:
# 주 비교 20/80; 중간 학습곡선은 [10,20,40,80]으로 바꾸되 별도 RUN_ID를 쓰세요.
from pathlib import Path
import importlib.util
try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False
MOUNT_DRIVE = IN_COLAB
RUN_ID = 'KSC_LLB_E14_matched_v1'
CFG = {
    'protocol_version': 'KSC-LLB-v1',
    'folds': [0,1,2,3,4],
    'budgets': [20,80],
    'label_seeds': [42,43,44,45,46],
    'head_ks': [1,2,4,8,16,32],
    'fixed_k': 4,
    'random_head_reps': 20,
    'label_shuffle_reps': 10,
    'bootstrap_reps': 5000,
    'noninferiority_margin': None,
    'implementation_sha256': 'c5cc5d43d0bbf0ced1c1620a798bfccaf8a7812dc6469d75aff9e884607cba0a',
}
RUN_CONTROLS = True
RUN_NEGATIVE_CONTROLS = True
CFG['run_controls'] = RUN_CONTROLS
CFG['run_negative_controls'] = RUN_NEGATIVE_CONTROLS
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
PROJECT_ROOT = Path('/content/drive/MyDrive/p8s') if MOUNT_DRIVE else Path.cwd() / 'p8s'
E14_ROOT = PROJECT_ROOT / 'E14_L13_head_5fold_v1'
WORK = Path('/content/ksc_llb_input') if IN_COLAB else Path.cwd() / 'ksc_llb_input'
OUT = PROJECT_ROOT / RUN_ID
print('입력 cache 후보:', E14_ROOT)
print('실험 결과:', OUT)

In [ ]:
from __future__ import annotations
import json, hashlib, time, warnings, platform
from pathlib import Path
import importlib.metadata
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (f1_score, balanced_accuracy_score, roc_auc_score,
                             precision_recall_fscore_support, confusion_matrix)
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits

STATE_NAMES = ['K+C+', 'K+C-', 'K-C-', 'K-C+']  # E14 order; do not silently reorder.
ROLE_NAMES = ['fit', 'utility_fit', 'policy_cal']
SOURCE_FILES = {
 'behavior': {'id':'1RXvW6o552ZNZn1PUMRaf8CEDMIUpqQOr', 'name':'p8s_behavior_v2.csv',
              'sha256':'fd37b0afd04972e3834ffa5f0af7f362affcfac8e20325117d37d8de15447c1c'},
 'bank': {'id':'1GK0xi217u57Wz_kn8ycg5eSlPgjsaFNy', 'name':'L13_neutral_head_bank_all_rows.npz',
          'sha256':'4ad4a040388d1e45995be7aadc7b91f6dfd1f4091b4fd452da8b4a9036b83802'},
 'full_oof': {'id':'1djoU4xPIAnCQjBLrb4a3YYIqStWLM6En', 'name':'OOF_L13_LAST_HIDDEN_PREDICTIONS.csv',
              'sha256':'2f8f214aa44fe1a6e020edbb1bf3456f9ed30aa147d9bce09b30bfa72150677d'},
 'head_oof': {'id':'1JRCiQGz3kEBz1Et2GrOUMmIhJjPcLRg3', 'name':'OOF_LOCKED_HEADSET_PREDICTIONS.csv',
              'sha256':'5e68831a5fb037b3f51855867befcd1dab58bd36263754a927332c9d82e23c1c'},
}

def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''): h.update(block)
    return h.hexdigest()

def save_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    tmp.replace(path)

def environment():
    result = {'python':platform.python_version()}
    for pkg in ['numpy','pandas','scipy','scikit-learn','threadpoolctl']:
        result[pkg] = importlib.metadata.version(pkg)
    return result

def freeze_plan(out, cfg):
    """Local, time-stamped analysis lock; NOT external preregistration."""
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    plan = {'config':cfg, 'environment':environment(), 'source_files':SOURCE_FILES,
            'scope':'controlled follow-up on previously inspected E14 data',
            'unit':'question with both C rows',
            'primary':'mean across seeds of pooled-OOF F1(20%)-F1(80%)',
            'bootstrap':'paired question resampling within original outer folds; fitted readers fixed',
            'label_budget':'unique labeled development questions including fit, utility_fit, policy_cal',
            'label_selection':'label-blind nested sampling within existing E14 roles',
            'no_claims':['independent confirmatory dataset','label-free learner','QA improvement','universal 4-head mechanism']}
    p=out/'ANALYSIS_PLAN.json'
    if p.exists():
        old=json.loads(p.read_text()); old.pop('created_utc', None)
        if old != plan: raise ValueError('Plan or environment changed. Choose a NEW RUN_ID; do not overwrite results.')
    else: save_json(p, {**plan,'created_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())})
    return file_sha(p)

## 2. 원본 입력 검증

원본 E14 노트북: `1ftHjJd26ACgkWo7ze2FS1W8cpUDScUUS` / `E14_L13_HeadDissection_5Fold.ipynb`.
모델은 `meta-llama/Llama-3.1-8B-Instruct`, 원본 특징 추출은 Transformers 4.57.1, Q→C 프롬프트, L13(0-based), batch 4였다. 이번 노트북은 **같은 npz의 정확한 bytes**만 사용하므로 extraction과 GPU 세팅을 새로 바꾸지 않는다.

파일 기반 라벨은 `kside_corr`이고, state 순서는 원 코드대로 `[K+C+, K+C-, K-C-, K-C+]`이다. 답변·정답·ctx_label·state·row id는 특징으로 입력하지 않는다. `row`는 원 cache 정렬 확인에만, `state`는 학습/평가에만 사용한다. cache를 다시 계산할 필요가 생기면 이 실험에서 조용히 대체하지 말고 별도 프로토콜로 분리한다.

In [ ]:
def obtain_sources(work, search_roots):
    """Reuse exact local/Drive bytes; otherwise download existing share IDs."""
    import shutil
    work=Path(work); work.mkdir(parents=True, exist_ok=True)
    paths={}
    for key, spec in SOURCE_FILES.items():
        dst=work/spec['name']
        candidates=[dst]+[Path(r)/spec['name'] for r in search_roots]
        good=next((p for p in candidates if p.is_file() and file_sha(p)==spec['sha256']),None)
        if good is not None:
            if good.resolve()!=dst.resolve(): shutil.copy2(good,dst)
        else:
            import gdown
            tmp=dst.with_name(dst.name+'.download')
            value=gdown.download(id=spec['id'],output=str(tmp),quiet=False)
            if value is None or not tmp.is_file():
                raise RuntimeError(f'{key} download failed. Mount the Drive account holding the E14 cache; no new dataset is needed.')
            if file_sha(tmp)!=spec['sha256']:
                raise ValueError(f'{key}: source hash differs. Stop; never substitute a newer file automatically.')
            tmp.replace(dst)
        if file_sha(dst)!=spec['sha256']: raise ValueError('Source hash mismatch: '+key)
        paths[key]=dst
        print('VERIFIED:',key,dst.name)
    return paths

def cell_ids(k,c):
    k,c=np.asarray(k,int),np.asarray(c,int)
    return np.where(k==1,np.where(c==1,0,1),np.where(c==1,3,2)).astype(int)

def build_outer_map(frame,seed=42):
    # Kept from original E14. The existing K-stratified benchmark split is frozen.
    q=frame[['question','k']].drop_duplicates().sort_values('question').reset_index(drop=True)
    rng=np.random.default_rng(seed); mapping={}
    for cls in (0,1):
        ids=q.loc[q.k.eq(cls),'question'].to_numpy().copy(); rng.shuffle(ids)
        mapping.update({s:i%5 for i,s in enumerate(ids)})
    return mapping

def load_e14(paths):
    f=pd.read_csv(paths['behavior']).reset_index(drop=True)
    f['k']=f.kside_corr.map({'K+':1,'K-':0,'K−':0})
    f['c']=f.ctx_label.map({'C+':1,'C-':0,'C−':0})
    if f[['question','ctx_text','k','c']].isna().any().any(): raise ValueError('Missing data or unrecognized labels.')
    f[['k','c']]=f[['k','c']].astype(int)
    f['state']=cell_ids(f.k,f.c); f['row']=np.arange(len(f))
    qnames=sorted(f.question.unique()); qmap={q:i for i,q in enumerate(qnames)}
    f['qid']=f.question.map(qmap).astype(int)
    f['outer_fold']=f.question.map(build_outer_map(f)).astype(int)
    assert len(f)==1224 and len(qnames)==612
    assert f.groupby('question').size().eq(2).all()
    assert f.groupby('question').k.nunique().eq(1).all()
    assert f.groupby('question').c.nunique().eq(2).all()
    assert f.state.value_counts().reindex(range(4)).tolist()==[362,362,250,250]
    refs={}
    for key in ['full_oof','head_oof']:
        r=pd.read_csv(paths[key]).sort_values('row').reset_index(drop=True)
        assert np.array_equal(r.row,f.row) and np.array_equal(r.true_state,f.state)
        assert np.array_equal(r.fold,f.outer_fold)
        refs[key]=r
    with np.load(paths['bank'],allow_pickle=False) as z:
        assert np.array_equal(z['rows'],f.row)
        bank={k:z[k].astype(np.float32) for k in ['BLOCK_LAST','HEAD_LAST','HEAD_Q','Q_MASS','C_MASS']}
    assert bank['BLOCK_LAST'].shape==(1224,4096)
    assert bank['HEAD_LAST'].shape==(1224,32,128)
    assert all(np.isfinite(v).all() for v in bank.values())
    return f,bank,refs

In [ ]:
PATHS = obtain_sources(WORK, [E14_ROOT, PROJECT_ROOT, Path.cwd(), Path('/mnt/data')])
frame, BANK, STORED_OOF = load_e14(PATHS)
PLAN_SHA = freeze_plan(OUT, CFG)
save_json(OUT / 'SOURCE_HASHES.json', {k:file_sha(p) for k,p in PATHS.items()})
print('질문:', frame.qid.nunique(), '입력:', len(frame))
display(frame.groupby(['k','c']).size().rename('rows').reset_index())
print('Analysis lock SHA256:', PLAN_SHA)

## 3. 고정 분할 + nested 라벨 예산

더 작은 예산에서 쓰는 질문은 큰 예산의 부분집합이며, `fit / utility_fit / policy_cal` 역할을 바꾸지 않는다. 80%에서는 원본 E14 split과 완전히 같다. 예산별 할당은 각 역할의 원래 크기를 따른 monotone proportional allocation으로 정한다. 같은 질문의 C± 두 행은 항상 같이 움직인다.

학습용 4개 라벨이 없는 조건은 코드가 실패를 보고한다. 실패한 시드를 제거해 좋은 시드만 평균내면 안 된다. 원본 고정 분할 복원 과정의 층화 라벨과 신규 budget 샘플링의 label-blind 선택을 구분한다.

In [ ]:
def original_roles(f,test_fold,seed=42):
    fold=f.outer_fold.to_numpy(); role=np.full(len(f),'fit',object)
    role[fold==test_fold]='test'; role[fold==(test_fold+1)%5]='cal'
    rng=np.random.default_rng(seed+1000*test_fold)
    for cls in (0,1):
        qs=np.array(sorted(set(f.loc[(role=='cal') & f.k.eq(cls),'question'])))
        rng.shuffle(qs); half=len(qs)//2
        role[(role=='cal') & f.question.isin(qs[:half])]='utility_fit'
        role[(role=='cal') & f.question.isin(qs[half:])]='policy_cal'
    assert 'cal' not in set(role)
    return role

def nested_counts(capacities,target):
    """Weighted fair allocation. Counts are monotone for every increasing budget."""
    cap=np.asarray(capacities,int); got=np.zeros(len(cap),int)
    if not (0<=target<=cap.sum()): raise ValueError('Invalid label budget')
    for _ in range(int(target)):
        ratio=np.where(got<cap,(got+1)/np.maximum(cap,1),np.inf)
        got[int(np.argmin(ratio))]+=1
    return got

def budget_roles(f,fold,budget,seed):
    base=original_roles(f,fold)
    byrole=[np.sort(f.loc[base==r,'qid'].unique()) for r in ROLE_NAMES]
    cap=np.array([len(q) for q in byrole])
    # 80 means all original development questions (489 or 490), not floor(0.8*N).
    n=int(cap.sum()) if budget==80 else int(np.floor(f.qid.nunique()*budget/100))
    counts=nested_counts(cap,n)
    role=np.full(len(f),'unused',object); role[base=='test']='test'
    for j,(name,qs,take) in enumerate(zip(ROLE_NAMES,byrole,counts)):
        rng=np.random.default_rng(np.random.SeedSequence([seed,fold,j,240]))
        order=rng.permutation(qs)  # No K/C labels inspected to select questions.
        role[f.qid.isin(order[:take])]=name
    for name in ROLE_NAMES:
        assert np.all(base[role==name]==name)
    assert np.array_equal(np.flatnonzero(base=='test'),np.flatnonzero(role=='test'))
    assert pd.DataFrame({'qid':f.qid,'role':role}).groupby('qid').role.nunique().eq(1).all()
    assert f.loc[np.isin(role,ROLE_NAMES),'qid'].nunique()==n
    if budget==80: assert np.array_equal(role,base)
    return role

def audit_all_splits(f,cfg,out):
    records=[]; manifests=[]
    for fold in cfg['folds']:
        for seed in cfg['label_seeds']:
            prev=set()
            for budget in sorted(cfg['budgets']):
                role=budget_roles(f,fold,budget,seed)
                used=set(f.loc[np.isin(role,ROLE_NAMES),'qid'])
                assert prev.issubset(used); prev=used
                manifest=f[['row','qid','question','outer_fold']].copy()
                manifest['role']=role; manifest['fold']=fold
                manifest['budget']=budget; manifest['label_seed']=seed
                manifests.append(manifest)
                for name in ROLE_NAMES+['test','unused']:
                    ix=np.flatnonzero(role==name)
                    record={'fold':fold,'budget':budget,'label_seed':seed,'role':name,
                            'questions':int(f.iloc[ix].qid.nunique()),'rows':len(ix)}
                    records.append(record)
                for name in ROLE_NAMES:
                    # Diagnostic after the budget is frozen. Never replace a failed seed.
                    if set(f.loc[role==name,'state'])!={0,1,2,3}:
                        save_json(Path(out)/'FAILED_SPLIT_AUDIT.json', {'fold':fold,'budget':budget,'seed':seed,'role':name,'status':'missing_class; no resampling permitted'})
                        raise ValueError(f'Predeclared seed has missing classes: {(fold,budget,seed,name)}. Record this failure; do not select a convenient seed.')
    pd.concat(manifests,ignore_index=True).to_csv(Path(out)/'SPLIT_AND_LABEL_AUDIT.csv',index=False)
    counts=pd.DataFrame(records); counts.to_csv(Path(out)/'LABEL_COUNTS.csv',index=False)
    return counts

In [ ]:
LABEL_AUDIT = audit_all_splits(frame, CFG, OUT)
display(LABEL_AUDIT.query('fold==0 and label_seed==42'))
assert sorted(CFG['folds']) == list(range(5)), '논문용 실행은 모든 fold가 필요합니다.'
assert 20 in CFG['budgets'] and 80 in CFG['budgets']

## 4. E14와 같은 판별기·선발 규칙

`fit`에서만 StandardScaler를 fit한다. Probe는 C=0.1, lbfgs, max_iter=2000, random_state=42, class_weight=None이다. **라벨 추출 시드만 42–46으로 바뀌며 Probe 시드는 고정한다.** 학습 경고를 숨기지 않는다.

헤드별 4-state probe → utility_fit의 **Balanced Accuracy, 동률이면 Macro-F1** 순으로 32개 헤드 순위 결정 → k∈{1,2,4,8,16,32} 각각을 fit으로 학습 → policy_cal의 **Balanced Accuracy, Macro-F1, 작은 k** 순으로 고정 → test 예측. 최종 선택 후 fit+cal을 합쳐 재학습하지 않는다(원 E14 경로 보존).

전체 표현을 4,096개 head 성분으로 자르는 것이 아니다. `HEAD_LAST`는 output projection 전의 head 벡터이고, `BLOCK_LAST`는 residual·MLP를 지난 블록 출력이므로 서로 다른 표현이다.

In [ ]:
def probe(X,y,fit):
    m=make_pipeline(StandardScaler(),LogisticRegression(C=0.1,max_iter=2000,
                         solver='lbfgs',random_state=42))
    with warnings.catch_warnings():
        warnings.simplefilter('error',ConvergenceWarning)
        with threadpool_limits(limits=1): m.fit(X[fit],y[fit])
    assert np.array_equal(m.classes_,np.arange(4))
    return m

def metrics(y,p):
    pred=p.argmax(1)
    out={'macro_f1':float(f1_score(y,pred,labels=range(4),average='macro',zero_division=0)),
         'bacc':float(balanced_accuracy_score(y,pred)),
         'accuracy':float(np.mean(y==pred))}
    _,rec,fs,_=precision_recall_fscore_support(y,pred,labels=range(4),zero_division=0)
    for j,n in enumerate(STATE_NAMES): out[n+'_f1']=float(fs[j]); out[n+'_recall']=float(rec[j])
    for key,cs in [('K',[0,1]),('C',[0,3])]:
        out[key+'_auroc']=float(roc_auc_score(np.isin(y,cs),p[:,cs].sum(1)))
    return out

def split_indices(role): return {r:np.flatnonzero(role==r) for r in ROLE_NAMES+['test']}

def rank_heads(H,y,ix):
    rows=[]
    for h in range(H.shape[1]):
        m=probe(H[:,h,:],y,ix['fit']); p=m.predict_proba(H[ix['utility_fit'],h,:])
        met=metrics(y[ix['utility_fit']],p)
        rows.append({'head':h,'four_BAcc_UF':met['bacc'],'four_macroF1_UF':met['macro_f1']})
    table=pd.DataFrame(rows)
    # Exactly E14's ordering, including the input head order used for ties.
    order=table.sort_values(['four_BAcc_UF','four_macroF1_UF'],ascending=False)['head'].astype(int).tolist()
    return order,table

def fit_headset(H,y,ix,order,ks=(1,2,4,8,16,32)):
    records=[]; models={}
    for k in ks:
        X=H[:,order[:k],:].reshape(len(H),-1)
        m=probe(X,y,ix['fit']); models[k]=m
        met=metrics(y[ix['policy_cal']],m.predict_proba(X[ix['policy_cal']]))
        records.append({'k':k,'bacc':met['bacc'],'macro_f1':met['macro_f1']})
    table=pd.DataFrame(records)
    best=table.sort_values(['bacc','macro_f1','k'],ascending=[False,False,True]).iloc[0]
    k=int(best['k']); return models[k],order[:k],table

def oof_record(f,test,probs,method,budget,seed,fold,rep=0):
    t=f.iloc[test][['row','qid','outer_fold']].copy()
    t['true_state']=f.state.to_numpy()[test]; t['pred_state']=probs.argmax(1)
    for j in range(4): t['p'+str(j)]=probs[:,j]
    t['method']=method; t['budget']=budget; t['label_seed']=seed; t['fold']=fold; t['rep']=rep
    return t

## 5. P1 — 라벨 예산만 바꾸는 직접 비교

80%는 각 fold에서 한 번만 학습한다. 20%는 fold마다 5개 nested subset 시드로 학습한다. 각 seed마다 1,224개 OOF 예측을 합쳐 Macro-F1을 계산하고, 그 5개 F1과 Δ를 보고한다. **5개 seed 확률의 평균을 내어 앙상블 성능으로 바꾸지 않는다.**

모든 폴더에 label manifest, head 순위, calibration 점수, 선택한 헤드와 OOF 예측을 남긴다. 같은 RUN_ID에서 설정이나 환경이 달라지면 `ANALYSIS_PLAN.json`이 막는다. 설정을 바꿀 때는 새로운 RUN_ID를 사용한다.

In [ ]:
def run_main(f,bank,cfg,out):
    out=Path(out); y=f.state.to_numpy(); H=bank['HEAD_LAST']; B=bank['BLOCK_LAST']
    outputs=[]; selections=[]
    for budget in cfg['budgets']:
        # Full-budget branch is deterministic; don't count copies as independent runs.
        seeds=[cfg['label_seeds'][0]] if budget==80 else cfg['label_seeds']
        for seed in seeds:
            for fold in cfg['folds']:
                root=out/f'main_b{budget}_s{seed}_f{fold}'; root.mkdir(exist_ok=True)
                ready=root/'COMPLETE.json'
                if ready.exists():
                    outputs.append(pd.read_csv(root/'oof.csv')); selections.append(json.loads((root/'selection.json').read_text())); continue
                role=budget_roles(f,fold,budget,seed); ix=split_indices(role)
                order,ranking=rank_heads(H,y,ix)
                m,heads,cal=fit_headset(H,y,ix,order,cfg['head_ks'])
                X=H[:,heads,:].reshape(len(H),-1); test=ix['test']
                hs=oof_record(f,test,m.predict_proba(X[test]),'head_adaptive',budget,seed,fold)
                fm=probe(B,y,ix['fit'])
                fs=oof_record(f,test,fm.predict_proba(B[test]),'full_hidden',budget,seed,fold)
                oo=pd.concat([hs,fs],ignore_index=True); oo.to_csv(root/'oof.csv',index=False)
                ranking.to_csv(root/'head_ranking.csv',index=False); cal.to_csv(root/'head_count_calibration.csv',index=False)
                selected={'fold':fold,'budget':budget,'label_seed':seed,'k':len(heads),'heads':heads,
                          'fit_questions':int(f.iloc[ix['fit']].qid.nunique()),
                          'label_questions':int(f.loc[np.isin(role,ROLE_NAMES),'qid'].nunique())}
                save_json(root/'selection.json',selected)
                save_json(ready,{'state':'completed','no_test_selection':True})
                outputs.append(oo); selections.append(selected)
                print('MAIN',budget,seed,fold,'heads',len(heads),flush=True)
    result=pd.concat(outputs,ignore_index=True); result.to_csv(out/'MAIN_OOF.csv',index=False)
    pd.DataFrame(selections).to_csv(out/'HEAD_SELECTIONS.csv',index=False)
    return result

def baseline_parity(main,refs,out):
    records=[]
    for method,key in [('full_hidden','full_oof'),('head_adaptive','head_oof')]:
        r=refs[key].sort_values('row'); n=main.query('budget==80 and method==@method').sort_values('row')
        if len(n)!=len(r): raise ValueError('Parity requires all five folds.')
        assert np.array_equal(n.row,r.row)
        now=metrics(n.true_state.to_numpy(),n[['p0','p1','p2','p3']].to_numpy())['macro_f1']
        old=float(f1_score(r.true_state,r.pred_state,average='macro'))
        records.append({'method':method,'new_macro_f1':now,'stored_macro_f1':old,
                        'delta':now-old,'argmax_agreement':float(np.mean(n.pred_state.to_numpy()==r.pred_state.to_numpy()))})
    table=pd.DataFrame(records); table.to_csv(Path(out)/'BASELINE_REPLAY_AUDIT.csv',index=False)
    # No automatic threshold forcing reproduction. New comparisons always use NEW 80% predictions.
    return table

In [ ]:
MAIN = run_main(frame, BANK, CFG, OUT)
REPLAY = baseline_parity(MAIN, STORED_OOF, OUT)
display(REPLAY)
print('저장된 E14 수치와 정확히 같지 않으면 BASELINE_REPLAY_AUDIT.csv를 논문 검증 기록에 남기세요.')
print('20%와 비교할 상대는 위에서 다시 학습한 80% 결과입니다. 저장된 0.6306을 강제로 넣지 않습니다.')

## 6. P2·P3·P4 — 주장별 대조군

이 절은 80% 조건에서 **k=4를 사전에 고정**한다. 본실험의 가변 k와 다른 질문을 평가하지 않는다. 순위 상위4개는 각 fold의 development 데이터에서 선정한다. H20/H10/H19/H3을 전체 OOF 결과로 고정하고 같은 질문에서 재검증하지 않는다.

**P2:** top4 vs uniform random4를 같은 512차원, 같은 reader로 비교한다. 무작위 헤드는 전체32개에서 고르며 selected와 겹쳐도 그대로 사용한다. 20회 중 test 최고점을 선택하지 않고 각 회차를 모두 기록한다. 전32개 HEAD_LAST(4,096차원)와 비교하여 residual 표현 차이와 차원 축소를 구분한다.

**P3:** 고정된 top4에 raw vector / norm(4차원) / head별 L2-unit vector(512차원)를 각각 동일 LR로 읽는다. 단위벡터 성능은 magnitude 없이도 판별 가능한 정보의 근거다. norm과 vector는 차원이 다르므로 성능 차이를 인과적인 방향 효과로 단정하지 않는다. raw vector로 고른 head에 대한 조건부 readout 비교이며, norm에 최적화된 다른 head를 탐색한 실험이 아니다.

**P4:** question-only와 question+context TF-IDF를 추가한다. 어휘·IDF는 fit에서만 학습한다. 고정 C=0.1, ngram(1,2), min_df=2, max_features=5000. gold·라벨·답변 문자열·예측결과는 텍스트 특징에 넣지 않는다. 두 텍스트 baseline과의 차이는 표현력뿐 아니라 모델 사전학습의 차이도 포함한다.

In [ ]:
def text_probe(text,y,ix):
    vec=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=5000)
    xfit=vec.fit_transform(np.asarray(text)[ix['fit']])
    m=LogisticRegression(C=0.1,max_iter=2000,solver='lbfgs',random_state=42)
    with warnings.catch_warnings():
        warnings.simplefilter('error',ConvergenceWarning)
        with threadpool_limits(limits=1): m.fit(xfit,y[ix['fit']])
    return m.predict_proba(vec.transform(np.asarray(text)[ix['test']]))

def run_controls(f,bank,cfg,out):
    """Control budget is fixed at 80%; k=4 fixed before seeing test scores."""
    y=f.state.to_numpy(); H=bank['HEAD_LAST']; out=Path(out); results=[]
    for fold in cfg['folds']:
        root=out/f'controls_f{fold}'; root.mkdir(exist_ok=True)
        if (root/'COMPLETE.json').exists(): results.append(pd.read_csv(root/'oof.csv')); continue
        ix=split_indices(budget_roles(f,fold,80,42)); test=ix['test']
        order,_=rank_heads(H,y,ix); heads=order[:cfg['fixed_k']]
        raw=H[:,heads,:]; norms=np.linalg.norm(raw,axis=2)
        unit=raw/np.maximum(norms[:,:,None],1e-8)
        variants={'top4_vector':raw.reshape(len(H),-1),
                  'top4_norm':norms,'top4_unit':unit.reshape(len(H),-1),
                  'all32_vector':H.reshape(len(H),-1)}
        rr=[]
        for name,X in variants.items():
            m=probe(X,y,ix['fit'])
            rr.append(oof_record(f,test,m.predict_proba(X[test]),name,80,42,fold))
        for rep in range(cfg['random_head_reps']):
            rng=np.random.default_rng(np.random.SeedSequence([2701,rep,fold]))
            # Sample from ALL 32 heads; overlap with selected heads is allowed.
            hs=rng.choice(32,size=cfg['fixed_k'],replace=False)
            X=H[:,hs,:].reshape(len(H),-1); m=probe(X,y,ix['fit'])
            rr.append(oof_record(f,test,m.predict_proba(X[test]),'random4_vector',80,42,fold,rep))
        q=f.question.astype(str).to_numpy(); c=f.ctx_text.astype(str).to_numpy()
        for name,text in [('question_tfidf',q),('question_context_tfidf',np.char.add(np.char.add(q.astype(str),' Context: '),c.astype(str)))]:
            p=text_probe(text,y,ix); rr.append(oof_record(f,test,p,name,80,42,fold))
        # Baselines depend on FIT labels only, not test frequencies.
        prior=np.bincount(y[ix['fit']],minlength=4).astype(float); prior/=prior.sum()
        majority=np.zeros((len(test),4)); majority[:,prior.argmax()]=1
        rr.append(oof_record(f,test,majority,'majority',80,42,fold))
        for rep in range(cfg['random_head_reps']):
            rng=np.random.default_rng(np.random.SeedSequence([8101,rep,fold]))
            p=np.eye(4)[rng.integers(0,4,size=len(test))]
            rr.append(oof_record(f,test,p,'uniform_random',80,42,fold,rep))
        oo=pd.concat(rr,ignore_index=True); oo.to_csv(root/'oof.csv',index=False)
        save_json(root/'LOCK.json',{'k':cfg['fixed_k'],'heads':heads,'budget':80,'random_chosen_without_test':True})
        save_json(root/'COMPLETE.json',{'state':'completed'})
        results.append(oo); print('CONTROLS',fold,flush=True)
    result=pd.concat(results,ignore_index=True); result.to_csv(out/'CONTROLS_OOF.csv',index=False)
    return result

In [ ]:
CONTROLS = run_controls(frame, BANK, CFG, OUT) if RUN_CONTROLS else None

## 7. P4 추가 — 개발 라벨 셔플 negative control

각 development 역할 안에서 K를 질문쌍 단위로 섞고, C 방향을 질문별로 무작위 교환한다. 같은 질문에 K가 일관되고 C±가 하나씩 있다는 구조, 역할별 4-class 표본 수를 보존하면서 label-feature 연결을 끊는다. 그다음 **head ranking과 k 선택을 포함한 전체 파이프라인**을 다시 실행한다. test의 참 라벨은 바꾸지 않고 점수를 계산한다.

이것은 누출·과도한 선택 편향에 대한 **진단용 negative control**이다. test까지 교환하는 정식 permutation null이 아니므로 `(넘은 횟수+1)/(반복+1)`을 유의확률로 보고하지 않는다. 10회 분포와 실제 판별기의 차이를 기술한다. 낮은 negative-control 점수만으로 모든 confounding이 제거되었다고 주장하지 않는다.

In [ ]:
def shuffled_dev_labels(f,role,seed):
    """Shuffle K at the question-pair level and independently randomize the C assignment.
    Within each role the exact four-class counts and paired-question structure survive.
    TEST remains untouched. This is a diagnostic negative control, NOT an exact p-test.
    """
    y=f.state.to_numpy().copy(); rng=np.random.default_rng(seed)
    for name in ROLE_NAMES:
        qs=np.sort(f.loc[role==name,'qid'].unique())
        pairs=np.array([f.index[(f.qid==q)&(role==name)].to_numpy()[np.argsort(f.loc[(f.qid==q)&(role==name),'c'].to_numpy())] for q in qs])
        labels=y[pairs].copy(); labels=labels[rng.permutation(len(labels))]
        swaps=rng.integers(0,2,len(labels)).astype(bool); labels[swaps]=labels[swaps,::-1]
        y[pairs]=labels
        original=f.state.to_numpy()[np.flatnonzero(role==name)]
        assert np.array_equal(np.bincount(original,minlength=4),np.bincount(y[role==name],minlength=4))
    return y

def run_negative_controls(f,bank,cfg,out):
    out=Path(out); H=bank['HEAD_LAST']; results=[]
    for rep in range(cfg['label_shuffle_reps']):
        for fold in cfg['folds']:
            root=out/f'negative_r{rep}_f{fold}'; root.mkdir(exist_ok=True)
            if (root/'COMPLETE.json').exists(): results.append(pd.read_csv(root/'oof.csv')); continue
            role=budget_roles(f,fold,80,42); ix=split_indices(role)
            y=shuffled_dev_labels(f,role,90000+100*rep+fold)
            order,_=rank_heads(H,y,ix); m,heads,_=fit_headset(H,y,ix,order,cfg['head_ks'])
            X=H[:,heads,:].reshape(len(H),-1)
            oo=oof_record(f,ix['test'],m.predict_proba(X[ix['test']]),'shuffled_dev_head_adaptive',80,42,fold,rep)
            oo.to_csv(root/'oof.csv',index=False)
            save_json(root/'COMPLETE.json',{'state':'completed','purpose':'negative control; not a formal permutation p-value'})
            results.append(oo); print('NEGATIVE',rep,fold,flush=True)
    result=pd.concat(results,ignore_index=True); result.to_csv(out/'NEGATIVE_OOF.csv',index=False)
    return result

In [ ]:
NEGATIVE = run_negative_controls(frame, BANK, CFG, OUT) if RUN_NEGATIVE_CONTROLS else None

## 8. 통계 분석 — 612개 질문이 기본 단위

주 estimand는 `mean_seed[pooled OOF F1(20%) − pooled OOF F1(80%)]`이다. 각 질문의 두 C행을 함께, 원래 outer test fold 안에서 복원추출한다. 같은 bootstrap 질문 가중치를 모든 조건과 seed에 동시에 적용한다. 5,000회 분포의 2.5/97.5 분위수를 보고한다. seed 간 점수 표준편차는 별도로 보고한다.

이 CI는 **이미 학습된 OOF 판별기와 고정된 5개 subset seed에 조건부**이다. bootstrap 때 전체 모델을 재학습한 CI가 아니며, 겹치는 훈련셋의 학습 불확실성까지 완전히 반영하지 않는다. 25개 fold×seed를 독립 표본으로 t-test하지 않는다. seed 확률 평균도 내지 않는다.

주 비교는 P1 하나다. P2–P4의 여러 CI는 탐색적 보조 결과이며 다중 비교를 고려하지 않은 채 각각 ‘유의하게 우월’하다고 결론내리지 않는다. formal superiority를 중심 주장으로 올릴 경우 별도 사전 분석계획과 multiplicity 보정이 필요하다.

**판정:** CI가 넓으면 ‘동등성이 입증되지 않았다’가 아니라 먼저 실제 감소량과 불확실성을 보고한다. 비열등성은 미리 정한 운영상 허용손실이 있을 때만 판정하며, 아래 기본설정에서는 판단을 만들지 않는다. 허용손실을 충족하지 못해도 실험 결과를 삭제하지 않는다.

In [ ]:
def summarize_oof(oof,out,filename):
    rows=[]
    for keys,g in oof.groupby(['method','budget','label_seed','rep'],sort=True):
        g=g.sort_values('row')
        if len(g)!=1224 or g.row.nunique()!=1224: raise ValueError('Incomplete or duplicated OOF rows.')
        p=g[['p0','p1','p2','p3']].to_numpy()
        rows.append(dict(zip(['method','budget','label_seed','rep'],keys))|metrics(g.true_state.to_numpy(),p))
    table=pd.DataFrame(rows); table.to_csv(Path(out)/filename,index=False)
    return table

def conditional_paired_ci(f,refs_a,refs_b,n_boot=5000,seed=7301):
    """Average the F1 differences across runs, not averaged probabilities (no ensemble).
    CIs are conditional on these fitted OOF readers and these label subset seeds.
    Each bootstrap resamples questions within the ORIGINAL OUTER TEST FOLDS.
    """
    if len(refs_a)!=len(refs_b): raise ValueError('Runs must be paired.')
    pred_a=[]; pred_b=[]
    for a,b in zip(refs_a,refs_b):
        a=a.sort_values('row'); b=b.sort_values('row')
        assert np.array_equal(a.row,f.row) and np.array_equal(b.row,f.row)
        assert np.array_equal(a.true_state,f.state) and np.array_equal(b.true_state,f.state)
        pred_a.append(a.pred_state.to_numpy()); pred_b.append(b.pred_state.to_numpy())
    y=f.state.to_numpy(); A=np.array(pred_a); B=np.array(pred_b)
    def macro(y,p,w=None):
        cm=np.bincount(4*y+p,weights=w,minlength=16).reshape(4,4)
        den=cm.sum(0)+cm.sum(1)
        return float(np.mean(np.divide(2*np.diag(cm),den,out=np.zeros(4),where=den!=0)))
    per_run=np.array([macro(y,a)-macro(y,b) for a,b in zip(A,B)])
    # Contribution of each question to each run's 4x4 confusion matrix.
    nq=f.qid.nunique(); qa=np.zeros((len(A),nq,16)); qb=np.zeros_like(qa)
    qid=f.qid.to_numpy()
    for r in range(len(A)):
        np.add.at(qa[r],(qid,4*y+A[r]),1); np.add.at(qb[r],(qid,4*y+B[r]),1)
    qfold=f.groupby('qid').outer_fold.first().reindex(range(nq)).to_numpy()
    strata=[np.flatnonzero(qfold==j) for j in range(5)]
    rng=np.random.default_rng(seed); distribution=[]
    for start in range(0,n_boot,100):
        n=min(100,n_boot-start); weights=np.zeros((n,nq),float)
        for st in strata:
            for i in range(n): weights[i,st]=rng.multinomial(len(st),np.full(len(st),1/len(st)))
        def scores(qcm):
            cm=np.einsum('bn,rnc->brc',weights,qcm).reshape(n,len(A),4,4)
            den=cm.sum(-2)+cm.sum(-1); diag=np.diagonal(cm,axis1=-2,axis2=-1)
            return np.divide(2*diag,den,out=np.zeros_like(diag),where=den!=0).mean(-1)
        distribution.extend((scores(qa)-scores(qb)).mean(1).tolist())
    lo,hi=np.quantile(distribution,[.025,.975])
    return {'delta_f1':float(per_run.mean()),'ci95_low':float(lo),'ci95_high':float(hi),
            'label_or_control_run_sd':float(per_run.std(ddof=1)) if len(per_run)>1 else 0.,
            'n_unique_questions':int(nq),'n_paired_runs':len(A),'bootstrap_reps':n_boot,
            'ci_scope':'conditional on fitted OOF readers; no independent retraining uncertainty'},np.asarray(distribution)

def make_comparisons(f,main,controls,cfg,out):
    out=Path(out); results=[]
    high=main.query("budget==80 and method=='head_adaptive'")
    low=[main.query("budget==20 and method=='head_adaptive' and label_seed==@s") for s in cfg['label_seeds']]
    result,dist=conditional_paired_ci(f,low,[high]*len(low),cfg['bootstrap_reps'])
    result={'contrast':'PRIMARY_head_20_minus_80',**result}
    # A margin cannot be selected by looking at the observed difference.
    margin=cfg['noninferiority_margin']
    result['noninferiority_margin']=margin
    result['noninferiority_status']='NOT_TESTED_margin_not_prespecified' if margin is None else (
        'conditional_noninferiority_supported' if result['ci95_low']>-margin else 'not_established')
    results.append(result); np.save(out/'PRIMARY_BOOTSTRAP_DELTAS.npy',dist)
    full80=main.query("budget==80 and method=='full_hidden'")
    res,_=conditional_paired_ci(f,[high],[full80],cfg['bootstrap_reps'])
    results.append({'contrast':'SECONDARY_adaptive_head_minus_fullhidden_80',**res})
    if controls is not None:
        top=controls.query("method=='top4_vector'")
        randoms=[controls.query("method=='random4_vector' and rep==@r") for r in range(cfg['random_head_reps'])]
        res,_=conditional_paired_ci(f,[top]*len(randoms),randoms,cfg['bootstrap_reps'])
        results.append({'contrast':'SECONDARY_top4_minus_random4_mean',**res})
        for method in ['top4_norm','top4_unit','all32_vector','question_context_tfidf']:
            other=controls.query('method==@method')
            res,_=conditional_paired_ci(f,[top],[other],cfg['bootstrap_reps'])
            results.append({'contrast':'SECONDARY_top4_minus_'+method,**res})
    table=pd.DataFrame(results); table.to_csv(out/'PAIRED_COMPARISONS.csv',index=False)
    return table

In [ ]:
MAIN_SUMMARY = summarize_oof(MAIN, OUT, 'MAIN_POOLED_BY_SEED.csv')
CONTROL_SUMMARY = summarize_oof(CONTROLS, OUT, 'CONTROLS_POOLED_BY_DRAW.csv') if CONTROLS is not None else None
NEGATIVE_SUMMARY = summarize_oof(NEGATIVE, OUT, 'NEGATIVE_POOLED_BY_DRAW.csv') if NEGATIVE is not None else None
PAIRED = make_comparisons(frame, MAIN, CONTROLS, CFG, OUT)
display(MAIN_SUMMARY)
display(PAIRED)

## 9. 논문 표·결과 문장 및 그림 생성

`PAPER_TABLE_BUDGETS.csv`: 같은 프로토콜의 20/80 결과. 80%는 결정론적인 1회/5-fold이며 seed SD를 인위적으로 만들지 않는다.  
`PAIRED_COMPARISONS.csv`: ΔF1·조건부 CI·적용된 허용치 여부.  
`PAPER_TABLE_CONTROLS.csv`: random4, unit/norm, 텍스트 baseline.  
`MAIN_POOLED_BY_SEED.csv`: 개별 상태 F1/recall 및 K/C AUROC.  
`PAPER_RESULT_SENTENCES.md`: 관측값만 들어간 결과 문장. **성공을 가정한 문장을 자동 출력하지 않는다.**

논문 초안의 `[P1: ...]`, `[P2: ...]`를 이 파일들과 대조하여 바꾼다. E16의 0.6120은 역사적 별도 실험으로만 남기고 새20% 칸에 옮기지 않는다.

In [ ]:
def export_paper_tables(main_summary,control_summary,negative_summary,comparisons,out):
    out=Path(out)
    group=main_summary.groupby(['method','budget']).agg(
        macro_f1_mean=('macro_f1','mean'),macro_f1_sd_across_label_seeds=('macro_f1','std'),
        runs=('macro_f1','size')).reset_index()
    group.to_csv(out/'PAPER_TABLE_BUDGETS.csv',index=False)
    parts=[t for t in [control_summary,negative_summary] if t is not None]
    if parts:
        pd.concat(parts,ignore_index=True).groupby('method').agg(macro_f1_mean=('macro_f1','mean'),
            control_draw_sd=('macro_f1','std'),draws=('macro_f1','size')).reset_index().to_csv(out/'PAPER_TABLE_CONTROLS.csv',index=False)
    p=comparisons.iloc[0]
    budgetrow=group.query("method=='head_adaptive'").set_index('budget')
    report=(f"# 같은 파이프라인 라벨 예산 비교 — 자동 집계\n\n"
            f"80%: {budgetrow.loc[80,'macro_f1_mean']:.4f}; "
            f"20%: {budgetrow.loc[20,'macro_f1_mean']:.4f}.\n\n"
            f"Δ(20−80)={p.delta_f1:.4f}, 조건부 95% CI "
            f"[{p.ci95_low:.4f}, {p.ci95_high:.4f}].\n\n"
            "같은 612개 질문에 대한 시드별 pooled OOF F1을 평균했다. "
            "평가 질문과 C± 두 행은 대응되며 시드 예측을 앙상블하지 않았다. "
            "신뢰구간은 고정된 학습 결과에 조건부인 질문 부트스트랩이다. "
            "비유의성을 동등성으로 해석하지 않는다. "
            "E14 데이터를 재사용한 통제 재평가이며 새로운 독립 데이터 검증은 아니다.\n\n"
            f"비열등성 상태: {p.noninferiority_status}.\n")
    (out/'PAPER_RESULT_SENTENCES.md').write_text(report,encoding='utf-8')
    return group,report

In [ ]:
PAPER_TABLE, RESULT_TEXT = export_paper_tables(MAIN_SUMMARY, CONTROL_SUMMARY, NEGATIVE_SUMMARY, PAIRED, OUT)
display(PAPER_TABLE)
print(RESULT_TEXT)
# 각 run에서 평가 행이 정확히 한 번 등장하는지 마지막 확인.
assert MAIN.groupby(['method','budget','label_seed','rep']).row.nunique().eq(1224).all()
save_json(OUT / 'FINAL_STATUS.json', {
    'status': 'completed', 'source_plan_sha256': PLAN_SHA,
    'run_controls': RUN_CONTROLS, 'run_negative_controls': RUN_NEGATIVE_CONTROLS,
    'not_measured': ['external generalization','causal LLM intervention','QA EM improvement'],
})

In [ ]:
# 그림은 조건별 별도의 figure로 저장. 평균확률 앙상블을 만들지 않습니다.
import matplotlib.pyplot as plt
for method in ['head_adaptive','full_hidden']:
    z=MAIN_SUMMARY.query('method==@method').groupby('budget').macro_f1.agg(['mean','std']).sort_index()
    fig,ax=plt.subplots(figsize=(6,4))
    ax.errorbar(z.index,z['mean'],yerr=z['std'].fillna(0),marker='o',capsize=4)
    ax.set_xlabel('Labeled questions (% of 612)');ax.set_ylabel('Pooled OOF macro-F1')
    ax.set_title(method+' (error bars: label-subset seed SD)')
    ax.set_xticks(z.index);fig.tight_layout();fig.savefig(OUT/(method+'_budget_curve.png'),dpi=200)
    plt.show()
for budget in [20,80]:
    s=CFG['label_seeds'][0]
    z=MAIN.query("method=='head_adaptive' and budget==@budget and label_seed==@s").sort_values('row')
    cm=confusion_matrix(z.true_state,z.pred_state,labels=range(4))
    fig,ax=plt.subplots(figsize=(5,4));ax.imshow(cm)
    ax.set_xticks(range(4),STATE_NAMES);ax.set_yticks(range(4),STATE_NAMES)
    for i in range(4):
        for j in range(4):ax.text(j,i,str(cm[i,j]),ha='center',va='center')
    ax.set_xlabel('Predicted');ax.set_ylabel('True')
    ax.set_title(f'Head probe / {budget}% / label seed {s}')
    fig.tight_layout();fig.savefig(OUT/f'confusion_b{budget}_seed{s}.png',dpi=200);plt.show()

## 10. P5 — 외부 검증 / QA 주장을 위한 별도 실행 설계

### P5-A. 독립된 새 질문에서 검증 (현재 입력 파일에는 없음)
같은 E14 질문으로 여러 번 실험한 것은 독립 검증이 아니다. 본 실험 완료 후 **평가하지 않은 새 질문 집합**을 확보한다. 정확한 질문 문자열 중복뿐 아니라 entity·relation 중복과 near-duplicate를 감사하고 공개한다. 샘플 수는 효과크기를 새 test에서 탐색하여 정하지 말고, 모집 가능 규모 또는 development 기반 CI 폭 계획으로 먼저 고정한다.

개입 없는 동일 prompt/model/L13 추출, 같은 K 채점 규약을 사용한다. 외부 C±의 생성 방식과 오류 유형을 고정하고 진위를 검수한다. **seed42·fold0의 20% 및 80% reader**를 미리 고정해 적용하는 평가를 기본안으로 삼는다. 기존 자료 전부로 재학습하거나 k를 다시 고르는 별도 설정과 섞지 않는다. 외부 결과를 본 뒤 head·threshold·C를 다시 고르지 않는다. 판독기별 새 질문 F1·상태별 recall·paired question CI를 보고하고, 이 실험이 완료되기 전에는 ‘도메인 일반화’라고 쓰지 않는다.

### P5-B. 실제 라우팅·정답 개선까지 주장하려는 경우 (이번 노트북 범위 밖)
4-state F1은 routing·steering 성능의 증명이 아니다. 추가 QA 실험에서는 고정한 action set의 결과를 학습/개발/평가 질문별로 준비하고, 같은 test에서 **무개입 A0, development에서 고른 최선의 고정 action, 예측 state router, gold-state router(진단용 상한), per-example action oracle(정답을 이용한 비실용 상한)**를 분리한다. 고정 action보다 router가 낫다는 비교는 router의 학습·선택을 development에서 끝낸 후 실시한다.

EM과 Recovery/Damage/Net, K+C−와 K−C+별 변화, intervention rate와 latency를 보고한다. 기존 E14의 `+2/1,224`를 이 추가 실험의 결과로 재사용하지 않는다. gold-state router가 좋아도 예측 router 개선은 입증되지 않는다. oracle은 실용 baseline이 아니며 test action 결과를 이용한 policy 선택은 누출이다. full-hidden vs head-set의 입력 차원 절감도 LLM 전체 FLOPs 절감과 다르다.

## 11. 결과에 따른 논문 주장 규칙

| 관측 결과 | 사용할 표현 | 사용하면 안 되는 표현 |
|---|---|---|
| P1: 감소량 작고 불확실성도 보고됨 | ‘개발 라벨을 약75% 줄였을 때 F1 변화는 Δ였다’ | ‘동등성 증명’, ‘총 학습 비용75% 절감’ |
| P1: 손실 큼 또는 CI 넓음 | 감소량을 보고하고 라벨 효율성의 범위를 제한 | 불리한 시드·fold 삭제 |
| P2: top4가 random4와 구분 안 됨 | 내부 head 표현에 신호가 있음 | ‘정예4개 선정의 이점 증명’ |
| P2: top4가 all32보다 낮음 | trade-off와512차원 결과를 함께 보고 | 모든 fold에서4개로 충분 |
| P3: unit와 raw가 비슷, norm이 낮음 | magnitude 외 정보로 판별 가능한 결과 | 특정 인과적 방향의 존재 확정 |
| P4: 텍스트 baseline도 높음 | 표면 정보와 내부 표현의 추가 이득을 분리 | 내부에만 고유한 신호라고 단정 |
| P4: shuffled-dev도 높음 | 파이프라인·정보누출 점검을 우선 | 새로운 일반화 성과라고 보고 |

## 12. 자료 출처와 작성 시 구분

**사용자 자료:** `붙여넣은 텍스트(1)(7).txt` E14/E16 실험 기록; `alltology_midterm_reordered (1).pdf` 26·52·54·55·58·59쪽; `DocForm_2.docx` 출판용 양식. 소량 라벨의 과거 기록과 새 실험은 별도다.

**원 구현·입력:** 위 SOURCE_FILES와 E14 원본 Colab 노트북. 이 문서의 budget 통제·대조군·통계 절차는 **추가로 설계한 실험**이며 기존에 수행했다고 서술하지 않는다.

**구현·분석을 확인한 외부 자료(논문 참고문헌에 자동 추가하지 않음):**
- scikit-learn 1.6.1 LogisticRegression 공식 문서: https://scikit-learn.org/1.6/modules/generated/sklearn.linear_model.LogisticRegression.html
- Dror et al., The Hitchhiker’s Guide to Testing Statistical Significance in Natural Language Processing, ACL 2018: https://aclanthology.org/P18-1128/ . 표본 단위·검정 절차 확인용이며, 여기의 구체적 conditional bootstrap 구현은 본 추가 설계이다.

요청에 따라 이전 초안의 Mallen et al.[6]와 Hewitt & Liang[7]은 새 논문 참고문헌에 넣지 않는다. 다만 최종 원고에는 원 질문 데이터 출처·이용 조건을 사실대로 기술해야 하며, 그 정보가 현재 자료만으로 불명확한 경우 임의로 생성하지 않는다.